# Lab: Structure Unstructured Restaurant Data with an LLM
**Module 1, Exercise 1 — IBM watsonx Submission**

## Install required libraries

In [ ]:
%%capture
%pip install numpy==2.3.4
%pip install matplotlib==3.10.7
%pip install ibm-watsonx-ai==1.4.7

## Import required libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import os

# IBM WatsonX imports
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import (
    ModelTypes,
    DecodingMethods,
)

# Suppress warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

## Fetch the data file

In [ ]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/1r_mM6ZPYNxcFv65QkzubA/California-Culinary-Map.txt

## Exercise 1: Load the data and display the texts
### Step 1: Load the data

In [ ]:
### 1.1: Define the file_path to the text file
file_path = "California-Culinary-Map.txt"

### 1.2: Open the text file
with open(file_path, 'r') as f:
    restaurant_data = f.read()

### 1.3: Print the first 100 characters of the restaurant data
print(restaurant_data[:100])

### Step 2: Split the restaurant paragraphs into a Python list

In [ ]:
### 2.1: Split the restaurant paragraphs into list
restaurant_list = restaurant_data.split("\n\n")

### 2.2: Since the first item is the dataset name, we remove it
restaurant_list = restaurant_list[1:]

### 2.3: Print out the number of restaurants we have
print(f"Number of restaurants: {len(restaurant_list)}")

### 2.4: Print out the first item to have a closer look at the content
print(restaurant_list[0])

## Exercise 2: Define the LLM
### Step 1: Implement the LLM function

In [ ]:
def llm_model(system_msg, prompt_txt):
    #system_msg: the system message given to the LLM
    #prompt_txt: the user prompt

    model_id = "ibm/granite-4-h-small"
    project_id = "skills-network"

    credentials = Credentials(
                    url = "https://us-south.ml.cloud.ibm.com"
                    )

    ### 1.1: Define the model by ModelInference
    model = ModelInference(
        model_id=model_id,
        credentials=credentials,
        project_id=project_id
    )

    ### 1.2: Define the messages
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt_txt}
    ]

    ### 1.3: Generate the chat response and return it
    response = model.chat(messages=messages)
    return response["choices"][0]["message"]["content"]

### Step 2: Test your llm_model()

In [ ]:
system_msg = "You are a helpful assistant."
prompt_txt = "Which place is warmer in winter? Hawaii or Greenland?"
print(llm_model(system_msg=system_msg, prompt_txt=prompt_txt))

## Exercise 3: Prompt Engineering
### Step 1: Define the template

In [ ]:
EXAMPLE_RESTAURANT_PARAGRAPH = restaurant_list[1]
EXAMPLE_OUTPUT = """
{{
  "name": "Mar de Cortez",
  "location": "Santa Monica",
  "type": "casual taqueria",
  "food_style": "Baja-style seafood",
  "rating": 4.2,
  "price_range": 1,
  "signatures": [
      "shrimp tacos with mango-habanero salsa",
      "grilled fish burritos"
  ],
  "vibe": "salt-air energy",
  "environment": "a premier sun-drenched spot for open-air dining near the pier.",
  "shortcomings": []
}}
"""

def restaurant_data_structure_prompt_generation(restaurant_paragraph):
    base_system_msg = """You are a data extraction assistant. Extract structured information from restaurant descriptions and output valid JSON. Follow the exact schema provided. For price_range, convert dollar signs ($$, $$$) into an integer. Output ONLY the JSON object, nothing else."""

    base_user_prompt = f"""Task:
Extract structured restaurant data as a valid JSON object.

Restaurant description:
{restaurant_paragraph}

Example:
Input Restaurant Description: {EXAMPLE_RESTAURANT_PARAGRAPH}
Output:
{EXAMPLE_OUTPUT}
"""
    return base_system_msg, base_user_prompt

### Step 2: Test your prompts

In [ ]:
restaurant_paragraph = restaurant_list[0]
base_system_msg, base_user_prompt = restaurant_data_structure_prompt_generation(restaurant_paragraph)

test_response = llm_model(system_msg=base_system_msg, prompt_txt=base_user_prompt)
print(test_response)

### Step 3: Validate the LLM outputs

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional

class Restaurant(BaseModel):
    name: str
    location: str
    type: str
    food_style: str
    rating: Optional[float] = None
    price_range: Optional[int] = None
    signatures: List[str] = Field(default_factory=list)
    vibe: Optional[str] = None
    environment: str
    shortcomings: List[str] = Field(default_factory=list)

### 3.2: Validate the test_response
try:
    restaurant_data_obj = Restaurant.model_validate_json(test_response)
    print(f"Success! Validated: {restaurant_data_obj.name}")
except ValidationError as e:
    print(f"Validation failed: {e.json()}")

## Exercise 4: Structure all the restaurant data
### Step 1: Define auto-repair prompts

In [ ]:
def JSON_auto_repair_prompts(candidate_json_output, error_message):
    auto_repair_system_msg = """You are a JSON repair expert. Fix invalid JSON to match the required schema. Output ONLY the corrected JSON object, nothing else."""

    auto_repair_prompt = f"""The following JSON output is invalid. Please fix it based on the error message.

Invalid JSON output:
{candidate_json_output}

Error message:
{error_message}

Please output ONLY the corrected, valid JSON object."""

    return auto_repair_system_msg, auto_repair_prompt

### Step 2: Process all restaurants
**SCREENSHOT: M1_1_structure_for_loop.jpg**

In [ ]:
error_log = []
structured_restaurant_lists = []

for i, restaurant_paragraph in enumerate(restaurant_list):
    try:
        base_system_msg, base_user_prompt = restaurant_data_structure_prompt_generation(restaurant_paragraph)
        candidate_output = llm_model(system_msg=base_system_msg, prompt_txt=base_user_prompt)

        valid = False
        for retries in range(4):
            try:
                Restaurant.model_validate_json(candidate_output)
                valid = True
                break
            except ValidationError as e:
                if retries < 3:
                    error_log.append(f"[{i}] Repair {retries+1}/3")
                    repair_sys, repair_prompt = JSON_auto_repair_prompts(candidate_output, e.json())
                    candidate_output = llm_model(system_msg=repair_sys, prompt_txt=repair_prompt)

        if not valid:
            error_log.append(f"[{i}] SKIPPED")
            continue

        structured_restaurant_lists.append(candidate_output)

    except Exception as e:
        error_log.append(f"[{i}] ERROR: {e}")
        continue

    if (i+1)%20 == 0:
        print(f'{i+1}/{len(restaurant_list)} done')

print(f'ALL DONE!! {len(structured_restaurant_lists)} processed, {len(error_log)} errors')

### Step 3: Save the list to a JSON file

In [ ]:
### Print the 50th item
if len(structured_restaurant_lists) >= 50:
    print(structured_restaurant_lists[49])
else:
    print(f"Only {len(structured_restaurant_lists)} restaurants processed; cannot show 50th.")

In [ ]:
structured_restaurant_lists_json = [json.loads(response) for response in structured_restaurant_lists]

# Assign itemId to each restaurant
for i, response in enumerate(structured_restaurant_lists_json):
    response['itemId'] = 1000001 + i

filename = 'structured_restaurant_data.json'
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(structured_restaurant_lists_json, f, indent=4)

print(f'Saved {len(structured_restaurant_lists_json)} restaurants to {filename}')